In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import fbeta_score,classification_report, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Define features and target variable
features = [
    # encoded station/service
    'StationCode_TE',
    'Service:Type_Intercity',
    'Service:Type_Intercity direct',
    'Service:Type_Sprinter',
    # temporal
    'Hour_sin',
    'Hour_cos',
    'DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'IsWeekend',
    'RushHour',
    # operational context
    'StationTraffic',
    'Stop:Platform change',
    'arrival_scheduled',
    'departure_scheduled',
    # weather
    'Wind Direction',
    'Hourly Mean Wind Speed',
    'Wind Speed last 10 Minutes',
    'Max Wind Speed',
    'Temperature',
    'Dew Point temperature',
    'Sunshine Duration',
    'Global Radiation',
    'Precipitation Duration',
    'Precipitation Amount',
    'Air Pressure',
    'Horizontal Visibility',
    'Cloud Cover',
    'Humidity',
    'Fog',
    'Rainfall',
    'Snowfall',
    'Thunder',
    'Hail',
]

target = "is_cancelled"

In [3]:
# Prepare for chunked reading
cols = features + [target]
chunk_size = 1_000_000
sample_size = 1_000_000
random_state = 42

dtype_map = {col: "float32" for col in features}
dtype_map[target] = "int8"


# Read CSV in chunks
def chunk_reader(file_path):
    for chunk in pd.read_csv(
        file_path,
        usecols=cols,
        dtype=dtype_map,
        chunksize=chunk_size
    ):
        chunk = chunk.reindex(columns=cols, fill_value=0) # Ensure all columns are present

        X_chunk = chunk[features]
        y_chunk = chunk[target]

        yield X_chunk, y_chunk # Yield the chunk for processing
        
# Count classes
not_cancelled_total = 0
cancelled_total = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):
    not_cancelled_total += (y_chunk == 0).sum()
    cancelled_total += (y_chunk == 1).sum()

total_rows = not_cancelled_total + cancelled_total

print(f"Train not cancelled: {not_cancelled_total:,}")
print(f"Train cancelled: {cancelled_total:,}")
print(f"Train total:     {total_rows:,}")

# Decide sample sizes
not_cancelled_sample_size = int(sample_size * not_cancelled_total / total_rows)
cancelled_sample_size = sample_size - not_cancelled_sample_size

print(f"Sampling not cancelled: {not_cancelled_sample_size:,}")
print(f"Sampling cancelled: {cancelled_sample_size:,}")

Train not cancelled: 43,903,090
Train cancelled: 4,479,366
Train total:     48,382,456
Sampling not cancelled: 907,417
Sampling cancelled: 92,583


In [4]:
# Generate random numbers for sampling (default_rng is faster)
rng = np.random.default_rng(random_state)

X_not_cancelled_parts = []
y_not_cancelled_parts = []
X_cancelled_parts = []
y_cancelled_parts = []

not_cancelled_seen = 0
cancelled_seen = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):

    # Create masks for cancelled and not cancelled
    not_cancelled_mask = y_chunk == 0
    cancelled_mask = y_chunk == 1
    
    # Split the chunk into cancelled and not cancelled
    X_not_cancelled = X_chunk.loc[not_cancelled_mask]
    y_not_cancelled = y_chunk.loc[not_cancelled_mask]

    X_cancelled = X_chunk.loc[cancelled_mask]
    y_cancelled = y_chunk.loc[cancelled_mask]

    # sample fraction based on remaining needed / remaining available
    neg_remaining_needed = not_cancelled_sample_size - sum(len(p) for p in y_not_cancelled_parts)
    pos_remaining_needed = cancelled_sample_size - sum(len(p) for p in y_cancelled_parts)

    neg_remaining_total = not_cancelled_total - not_cancelled_seen
    pos_remaining_total = cancelled_total - cancelled_seen

    # Sample not cancelled
    if neg_remaining_needed > 0 and len(X_not_cancelled) > 0:
        p_neg = min(1.0, neg_remaining_needed / max(neg_remaining_total, 1))
        keep_neg = rng.random(len(X_not_cancelled)) < p_neg

        X_not_cancelled_parts.append(X_not_cancelled.loc[keep_neg])
        y_not_cancelled_parts.append(y_not_cancelled.loc[keep_neg])
        
    # Sample cancelled
    if pos_remaining_needed > 0 and len(X_cancelled) > 0:
        p_pos = min(1.0, pos_remaining_needed / max(pos_remaining_total, 1))
        keep_pos = rng.random(len(X_cancelled)) < p_pos

        X_cancelled_parts.append(X_cancelled.loc[keep_pos])
        y_cancelled_parts.append(y_cancelled.loc[keep_pos])

    not_cancelled_seen += len(X_not_cancelled)
    cancelled_seen += len(X_cancelled)

# Combine sampled parts
X_train_sample = pd.concat(X_not_cancelled_parts + X_cancelled_parts, axis=0)
y_train_sample = pd.concat(y_not_cancelled_parts + y_cancelled_parts, axis=0)

# Shuffle final sample
shuffle_idx = rng.permutation(len(X_train_sample))
X_train_sample = X_train_sample.iloc[shuffle_idx].to_numpy(dtype=np.float32, copy=False)
y_train_sample = y_train_sample.iloc[shuffle_idx].to_numpy(copy=False)

In [12]:
# Convert training sample to numpy arrays
X_values = np.asarray(X_train_sample, dtype=np.float32)
y_values = np.asarray(y_train_sample, dtype=np.int8)

# Calculate column means from training data only
column_means = np.nanmean(X_values, axis=0).astype(np.float32)
column_means = np.where(np.isnan(column_means), 0.0, column_means).astype(np.float32)

# Impute missing values in the training data
X_values = np.where(
    np.isnan(X_values),
    column_means,
    X_values
).astype(np.float32, copy=False)

# Compute class/sample weights
classes = np.array([0, 1], dtype=np.int8)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_values
)

class_weight_dict = {
    0: weights[0],
    1: weights[1]
}

# Define Logistic Regression using SGD
sgd = SGDClassifier(
    loss="log_loss",
    random_state=42
)

# 6. Train in chunks
chunk_size = 1_000_000

for i, start in enumerate(range(0, len(X_values), chunk_size), start=1):
    end = start + chunk_size

    X_chunk_np = X_values[start:end]
    y_chunk_np = y_values[start:end]

    sample_weights = np.array(
        [class_weight_dict[label] for label in y_chunk_np],
        dtype=np.float32
    )

    if i == 1:
        sgd.partial_fit(
            X_chunk_np,
            y_chunk_np,
            classes=classes,
            sample_weight=sample_weights
        )
    else:
        sgd.partial_fit(
            X_chunk_np,
            y_chunk_np,
            sample_weight=sample_weights
        )

    print(f"Train chunk {i}: {X_chunk_np.shape}")

print(f"Logistic Regression trained on {len(y_values):,} samples")

Train chunk 1: (999797, 35)
Logistic Regression trained on 999,797 samples


In [13]:
def find_best_threshold_lr(file_path, model):
    y_true_parts = []
    y_prob_parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        # Impute missing values using training-set column means
        if np.isnan(X_chunk_np).any():
            X_chunk_np = np.where(
                np.isnan(X_chunk_np),
                column_means,
                X_chunk_np
            ).astype(np.float32, copy=False)

        y_prob = model.predict_proba(X_chunk_np)[:, 1]

        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_prob_parts.append(y_prob)

    y_true_all = np.concatenate(y_true_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    best_threshold = 0.5
    best_f2 = -1

    for threshold in np.arange(0.01, 0.99, 0.01):
        y_pred_all = (y_prob_all >= threshold).astype(int)
        f2 = fbeta_score(y_true_all, y_pred_all, beta=2, zero_division=0)

        if f2 > best_f2:
            best_f2 = f2
            best_threshold = threshold

    print("Best validation threshold:", round(best_threshold, 2))
    print("Best validation F2:", round(best_f2, 4))

    return best_threshold

def evaluate_split_lr(name, file_path, model, threshold):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        # Impute missing values using training-set column means
        if np.isnan(X_chunk_np).any():
            X_chunk_np = np.where(
                np.isnan(X_chunk_np),
                column_means,
                X_chunk_np
            ).astype(np.float32, copy=False)

        y_prob = model.predict_proba(X_chunk_np)[:, 1]
        y_pred = (y_prob >= threshold).astype(int)

        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(y_pred)
        y_prob_parts.append(y_prob)

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    f2 = fbeta_score(y_true_all, y_pred_all, beta=2, zero_division=0)
    pr_auc = average_precision_score(y_true_all, y_prob_all)

    print(f"\n{name}")
    print("Threshold:", round(threshold, 2))
    print("F2 Score:", round(f2, 4))
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("PR-AUC:", round(pr_auc, 4))

    return {
        "threshold": threshold,
        "f2": f2,
        "pr_auc": pr_auc
    }

In [14]:
best_threshold = find_best_threshold_lr("val_data.csv", sgd)

train_results = evaluate_split_lr(
    name="Train",
    file_path="train_data.csv",
    model=sgd,
    threshold=best_threshold
)

val_results = evaluate_split_lr(
    name="Validation",
    file_path="val_data.csv",
    model=sgd,
    threshold=best_threshold
)

test_results = evaluate_split_lr(
    name="Test",
    file_path="test_data.csv",
    model=sgd,
    threshold=best_threshold
)

Best validation threshold: 0.01
Best validation F2: 0.3564

Train
Threshold: 0.01
F2 Score: 0.3378
              precision    recall  f1-score   support

           0      0.685     0.000     0.000  43903090
           1      0.093     1.000     0.169   4479366

    accuracy                          0.093  48382456
   macro avg      0.389     0.500     0.085  48382456
weighted avg      0.630     0.093     0.016  48382456

PR-AUC: 0.0926

Validation
Threshold: 0.01
F2 Score: 0.3564
              precision    recall  f1-score   support

           0      0.142     0.000     0.000   9333880
           1      0.100     1.000     0.181   1033789

    accuracy                          0.100  10367669
   macro avg      0.121     0.500     0.091  10367669
weighted avg      0.138     0.100     0.018  10367669

PR-AUC: 0.0997

Test
Threshold: 0.01
F2 Score: 0.4093
              precision    recall  f1-score   support

           0      0.420     0.000     0.000   9105889
           1      0.122 